# PDAC Survival Prediction — Evaluation & Results

**Author:** Kumar Mahat | Texas Tech University

## Final Results

| Metric | Published Benchmark | **This Work** |
|--------|-------------------|---------------|
| ACC | 0.850 | **0.878** |
| PPV | — | **0.918** |
| AUC-ROC | — | **0.903** |

> Exceeded the Nature Cancer (2024) published benchmark by **+0.028 ACC**

In [ ]:
!pip install pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn imbalanced-learn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix,
    roc_curve, precision_recall_curve
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')
print('Ready')

In [ ]:
# ============================================================
# LOAD DATA AND MODEL
# ============================================================

try:
    X = pd.read_csv('/content/drive/MyDrive/PDAC_Research/X_final_20features.csv')
    y = pd.read_csv('/content/drive/MyDrive/PDAC_Research/y_labels.csv').values.ravel()
    print(f'Data loaded: X={X.shape}, y={y.shape}')
except:
    print('Generating sample data (run previous notebooks for real results)')
    np.random.seed(42)
    X = pd.DataFrame(np.random.randn(150, 20), columns=[f'feature_{i}' for i in range(20)])
    y = np.random.randint(0, 2, 150)

# Preprocessing
imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
X_processed = scaler.fit_transform(imputer.fit_transform(X))

# Class balancing
smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_processed, y)
print(f'Balanced dataset: {X_balanced.shape}')

In [ ]:
# ============================================================
# FINAL MODEL EVALUATION WITH STRATIFIED K-FOLD
# ============================================================

# Best models from tuning
xgb_best = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='logloss', verbosity=0
)

lgb_best = LGBMClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    num_leaves=63, subsample=0.8,
    random_state=42, verbosity=-1
)

ensemble = VotingClassifier(
    estimators=[('xgb', xgb_best), ('lgb', lgb_best)],
    voting='soft'
)

# Collect predictions from all folds
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
all_y_true, all_y_pred, all_y_proba = [], [], []

for fold, (train_idx, test_idx) in enumerate(cv.split(X_balanced, y_balanced)):
    X_train, X_test = X_balanced[train_idx], X_balanced[test_idx]
    y_train, y_test = y_balanced[train_idx], y_balanced[test_idx]
    
    ensemble.fit(X_train, y_train)
    y_pred = ensemble.predict(X_test)
    y_proba = ensemble.predict_proba(X_test)[:, 1]
    
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_proba.extend(y_proba)
    
    fold_acc = accuracy_score(y_test, y_pred)
    print(f'Fold {fold+1} ACC: {fold_acc:.3f}')

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)
all_y_proba = np.array(all_y_proba)

In [ ]:
# ============================================================
# FINAL METRICS
# ============================================================

acc  = accuracy_score(all_y_true, all_y_pred)
ppv  = precision_score(all_y_true, all_y_pred)
rec  = recall_score(all_y_true, all_y_pred)
auc  = roc_auc_score(all_y_true, all_y_proba)

print('=' * 50)
print('FINAL RESULTS — Soft-Voting Ensemble (XGB + LGB)')
print('=' * 50)
print(f'ACC (Accuracy):           {acc:.3f}')
print(f'PPV (Precision):          {ppv:.3f}')
print(f'Sensitivity (Recall):     {rec:.3f}')
print(f'AUC-ROC:                  {auc:.3f}')
print()
print('Published Benchmark (Nature Cancer 2024):')
print(f'ACC:                      0.850')
print(f'Improvement:              +{acc - 0.850:.3f}')
print('=' * 50)

In [ ]:
# ============================================================
# VISUALIZATIONS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion Matrix
cm = confusion_matrix(all_y_true, all_y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Short-term', 'Long-term'],
            yticklabels=['Short-term', 'Long-term'])
axes[0].set_title('Confusion Matrix\n(5-Fold CV)')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(all_y_true, all_y_proba)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUC = {auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].axhline(y=0.850, color='orange', linestyle='--', alpha=0.7, label='Benchmark ACC=0.850')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1])

# 3. Benchmark comparison bar chart
metrics = ['ACC', 'PPV', 'AUC']
our_scores = [acc, ppv, auc]
benchmark = [0.850, None, None]

x = np.arange(len(metrics))
axes[2].bar(x - 0.2, our_scores, 0.35, label='This Work', color='steelblue')
axes[2].bar([0 + 0.2], [0.850], 0.35, label='Published Benchmark', color='orange', alpha=0.7)
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics)
axes[2].set_ylim(0.7, 1.0)
axes[2].set_ylabel('Score')
axes[2].set_title('Results vs Published Benchmark\n(Nature Cancer 2024)')
axes[2].legend()

for i, v in enumerate(our_scores):
    axes[2].text(i - 0.2, v + 0.005, f'{v:.3f}', ha='center', fontsize=10, fontweight='bold')
axes[2].text(0 + 0.2, 0.850 + 0.005, '0.850', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('PDAC Survival Prediction — Exceeding Nature Cancer (2024) Benchmark', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('final_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Results plot saved: final_results.png')

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print('Classification Report:')
print(classification_report(all_y_true, all_y_pred,
                           target_names=['Short-term (<12mo)', 'Long-term (>=12mo)']))

print('\nSummary:')
summary = pd.DataFrame({
    'Metric': ['ACC', 'PPV', 'AUC-ROC', 'Published Benchmark (ACC)', 'Improvement'],
    'Value': [f'{acc:.3f}', f'{ppv:.3f}', f'{auc:.3f}', '0.850', f'+{acc-0.850:.3f}']
})
print(summary.to_string(index=False))